## 0 · Setup
Imports and Snowflake connection.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, base64
import numpy as np
import pandas as pd
import polars as pl
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from snowflake import connector as snowflake_connector
from statsmodels.regression.linear_model import OLS

load_dotenv('.env')
conn = snowflake_connector.connect(**{
    'account'     : os.getenv('SNOWFLAKE_ACCOUNT'),
    'warehouse'   : os.getenv('SNOWFLAKE_WAREHOUSE'),
    'role'        : os.getenv('SNOWFLAKE_ROLE'),
    'user'        : os.getenv('SNOWFLAKE_USERNAME'),
    'private_key' : base64.b64decode(os.getenv('SNOWFLAKE_PRIVATE_KEY')),
})
DB = 'otc_intern_project.project_7'

def q(sql):
    return pl.read_database(sql, conn).select(pl.all().name.to_lowercase())

print('OK')

## 1 · Announcement Surprises
Loads the Bloomberg economic calendar and computes `surprise = actual − survey_median` for the 10 US macro releases used in Faust et al.

**Produces:** `us_cal` — one row per announcement with columns `label`, `release_date`, `surprise_adj`, `win_start`, `win_end` (UTC).

Sign convention: positive surprise = stronger-than-expected growth (countercyclical indicators Initial Claims and Unemployment are flipped).

In [ ]:
# ── Event map: Bloomberg name → (label, ET hour, ET min) ─────────────────────
EVENTS = {
    'CPI MoM'                          : ('CPI',              8, 30),
    'PPI Final Demand MoM'             : ('PPI',              8, 30),
    'GDP Annualized QoQ'               : ('GDP',              8, 30),
    'Change in Nonfarm Payrolls'       : ('Nonfarm Payrolls', 8, 30),
    'Unemployment Rate'                : ('Unemployment',     8, 30),
    'Initial Jobless Claims'           : ('Initial Claims',   8, 30),
    'Housing Starts'                   : ('Housing Starts',   8, 30),
    'Retail Sales Advance MoM'         : ('Retail Sales',     8, 30),
    'Trade Balance'                    : ('Trade Balance',    8, 30),
    'FOMC Rate Decision (Lower Bound)' : ('Fed Funds',       14,  0),
}
FLIP = {'Unemployment', 'Initial Claims'}
ET, UTC = ZoneInfo('America/New_York'), ZoneInfo('UTC')

def utc_window(d, h, m, pre=5, post=15):
    rel = datetime(d.year, d.month, d.day, h, m).replace(tzinfo=ET).astimezone(UTC)
    return rel - timedelta(minutes=pre), rel + timedelta(minutes=post)

# Load calendar
cal = (
    pl.read_excel('2020-01-01_to_2026-01-01_economic_calendar.xlsx')
    .select(pl.all().name.to_lowercase())
    .filter(pl.col('release_date').cast(pl.Date, strict=False).is_not_null())
    .select(['country_name','release_date','event_name','survey_median','actual'])
    .with_columns([
        pl.col('release_date').cast(pl.Date),
        pl.col('survey_median').cast(pl.Float64, strict=False),
        pl.col('actual').cast(pl.Float64, strict=False),
    ])
    .drop_nulls()
)

# Filter to US events, add label + sign-adjusted surprise + UTC windows
rows = []
for r in cal.filter(pl.col('country_name') == 'United States').iter_rows(named=True):
    if r['event_name'] not in EVENTS:
        continue
    label, h, m = EVENTS[r['event_name']]
    surprise = r['actual'] - r['survey_median']
    if label in FLIP:
        surprise = -surprise
    t0, t1 = utc_window(r['release_date'], h, m)
    rows.append({'label': label, 'release_date': r['release_date'],
                 'surprise_adj': surprise, 'win_start': t0, 'win_end': t1})

us_cal = pl.DataFrame(rows).sort('release_date')
print(us_cal.group_by('label').len().sort('label'))
us_cal.head(3)

## 2 · FX 20-Minute Window Returns
For each announcement, fetches the last 1-min TWAP bar before the window opens and the last bar before it closes, then computes log-return × 10,000 (basis points).

**Produces:** `fx_returns` — columns `label`, `release_date`, `surprise_adj`, `security`, `ccy`, `return_bp`.

Sign: **negative return = dollar appreciates** (Bloomberg CMPN is foreign-per-USD, so sign is flipped to match Faust's USD-per-foreign convention).

In [ ]:
FX_SECS = [
    'AUD CMPN Curncy','CAD CMPN Curncy','CHF CMPN Curncy','DKK CMPN Curncy',
    'EUR CMPN Curncy','GBP CMPN Curncy','JPY CMPN Curncy','NOK CMPN Curncy',
    'NZD CMPN Curncy','SEK CMPN Curncy',
]

# Build VALUES clause of announcement windows
vals = ',\n'.join(
    f"('{r['label']}','{r['release_date']}',"
    f"'{r['win_start'].strftime('%Y-%m-%d %H:%M:%S')}',"
    f"'{r['win_end'].strftime('%Y-%m-%d %H:%M:%S')}',"
    f"{r['surprise_adj']})"
    for r in us_cal.iter_rows(named=True)
)
windows_cte = f"""
    SELECT label, release_date::DATE AS release_date,
           win_start::TIMESTAMP_NTZ AS win_start,
           win_end::TIMESTAMP_NTZ   AS win_end,
           surprise_adj::FLOAT      AS surprise_adj
    FROM (VALUES {vals}) v(label,release_date,win_start,win_end,surprise_adj)
"""
sec_list = ','.join(f"'{s}'" for s in FX_SECS)

fx_sql = f"""
WITH windows AS ({windows_cte}),
prices AS (
    SELECT CONVERT_TIMEZONE('UTC', bin_start_time) AS ts, security, twap AS price
    FROM {DB}.intraday_fx_trades_1min
    WHERE security IN ({sec_list}) AND twap > 0
),
pre AS (
    SELECT w.label, w.release_date, w.surprise_adj, p.security, p.price,
           ROW_NUMBER() OVER (PARTITION BY w.label,w.release_date,p.security ORDER BY p.ts DESC) rn
    FROM windows w JOIN prices p
      ON p.ts >= DATEADD(minute,-30,w.win_start) AND p.ts < w.win_start
),
post AS (
    SELECT w.label, w.release_date, p.security, p.price,
           ROW_NUMBER() OVER (PARTITION BY w.label,w.release_date,p.security ORDER BY p.ts DESC) rn
    FROM windows w JOIN prices p
      ON p.ts >= w.win_start AND p.ts <= w.win_end
)
SELECT pr.label, pr.release_date, pr.surprise_adj, pr.security,
       LN(po.price / NULLIF(pr.price,0)) * 10000.0 * -1.0 AS return_bp
FROM pre pr JOIN post po USING (label,release_date,security)
WHERE pr.rn=1 AND po.rn=1 AND pr.price>0 AND po.price>0
ORDER BY pr.release_date, pr.label, pr.security
"""

fx_returns = q(fx_sql).with_columns(
    pl.col('security').str.split(' ').list.get(0).alias('ccy')
)
print(f'FX returns: {len(fx_returns):,} rows')
fx_returns.head(3)

## 3 · IRS 20-Minute Window Returns
Same logic as FX but for interest rate swaps. Change in swap rate × 100 = basis points.

**Produces:** `irs_returns` — columns `label`, `release_date`, `surprise_adj`, `security`, `ccy`, `tenor_yr`, `return_bp`.

Sign: **positive = rates rose** (no flip needed, unlike FX).

In [ ]:
irs_secs_df = q(f"SELECT DISTINCT security FROM {DB}.intraday_irs_trades_1min ORDER BY security")
IRS_SECS = irs_secs_df['security'].to_list()
irs_sec_list = ','.join(f"'{s}'" for s in IRS_SECS)

irs_sql = f"""
WITH windows AS ({windows_cte}),
prices AS (
    SELECT CONVERT_TIMEZONE('UTC', bin_start_time) AS ts, security, twap AS price
    FROM {DB}.intraday_irs_trades_1min
    WHERE twap IS NOT NULL
),
pre AS (
    SELECT w.label, w.release_date, w.surprise_adj, p.security, p.price,
           ROW_NUMBER() OVER (PARTITION BY w.label,w.release_date,p.security ORDER BY p.ts DESC) rn
    FROM windows w JOIN prices p
      ON p.ts >= DATEADD(minute,-30,w.win_start) AND p.ts < w.win_start
),
post AS (
    SELECT w.label, w.release_date, p.security, p.price,
           ROW_NUMBER() OVER (PARTITION BY w.label,w.release_date,p.security ORDER BY p.ts DESC) rn
    FROM windows w JOIN prices p
      ON p.ts >= w.win_start AND p.ts <= w.win_end
)
SELECT pr.label, pr.release_date, pr.surprise_adj, pr.security,
       (po.price - pr.price) * 100.0 AS return_bp
FROM pre pr JOIN post po USING (label,release_date,security)
WHERE pr.rn=1 AND po.rn=1
ORDER BY pr.release_date, pr.label, pr.security
"""

irs_returns = q(irs_sql).with_columns([
    pl.col('security').str.split(' ').list.get(0).alias('ccy'),
    pl.col('security').str.split(' ').list.get(1).cast(pl.Int32, strict=False).alias('tenor_yr'),
])
print(f'IRS returns: {len(irs_returns):,} rows | Tenors: {sorted(irs_returns["tenor_yr"].drop_nulls().unique().to_list())}')
irs_returns.head(3)

## 4 · Non-Announcement Baseline Returns
FX returns in the same 20-minute time slot (8:25–8:45 ET) on days with **no** US macro release. Used to compute the volatility ratio in Faust Table 4.

**Produces:** `fx_noann` — columns `day_`, `security`, `return_bp`.

In [ ]:
ann_dates = "(" + ",".join(f"'{d}'" for d in us_cal['release_date'].unique().to_list()) + ")"

noann_sql = f"""
WITH bars AS (
    SELECT CONVERT_TIMEZONE('UTC', bin_start_time) AS ts, security, twap AS price
    FROM {DB}.intraday_fx_trades_1min
    WHERE security IN ({sec_list})
      AND twap > 0
      AND DATE_TRUNC('day', CONVERT_TIMEZONE('UTC', bin_start_time))::DATE NOT IN {ann_dates}
      AND (
          (EXTRACT(HOUR FROM CONVERT_TIMEZONE('UTC',bin_start_time))=13
           AND EXTRACT(MINUTE FROM CONVERT_TIMEZONE('UTC',bin_start_time)) BETWEEN 25 AND 44)
          OR
          (EXTRACT(HOUR FROM CONVERT_TIMEZONE('UTC',bin_start_time))=12
           AND EXTRACT(MINUTE FROM CONVERT_TIMEZONE('UTC',bin_start_time)) BETWEEN 25 AND 44)
      )
),
daily AS (
    SELECT DATE_TRUNC('day',ts)::DATE AS day_, security,
           FIRST_VALUE(price) OVER (PARTITION BY DATE_TRUNC('day',ts),security ORDER BY ts
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS p_open,
           LAST_VALUE(price)  OVER (PARTITION BY DATE_TRUNC('day',ts),security ORDER BY ts
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS p_close
    FROM bars
)
SELECT DISTINCT day_, security,
       LN(p_close/NULLIF(p_open,0))*10000.0*-1.0 AS return_bp
FROM daily WHERE p_open>0 AND p_close>0
"""

fx_noann = q(noann_sql)
print(f'Non-ann baseline: {len(fx_noann):,} rows | {fx_noann["day_"].n_unique():,} days')
fx_noann.head(3)

## 5 · Helper: OLS Regression
Faust eq. (1): `return_bp = β × surprise_adj + ε` — no intercept, HC3 robust standard errors.

Call `faust_ols(df, event, group_col, group_val)` to get `{beta, se, t, p, r2, n}` for any event × asset slice.

In [ ]:
def faust_ols(df, event, group_col, group_val):
    sub = (
        df.filter((pl.col('label') == event) & (pl.col(group_col) == group_val))
          .drop_nulls(subset=['surprise_adj','return_bp'])
          .filter(pl.col('return_bp').is_finite())
    )
    if len(sub) < 8:
        return dict(beta=np.nan, se=np.nan, t=np.nan, p=np.nan, r2=np.nan, n=len(sub))
    y = sub['return_bp'].to_numpy()
    X = sub['surprise_adj'].to_numpy().reshape(-1,1)
    res = OLS(y, X).fit(cov_type='HC3')
    return dict(beta=res.params[0], se=res.bse[0], t=res.tvalues[0],
                p=res.pvalues[0], r2=max(1-res.ssr/np.sum(y**2), 0), n=int(res.nobs))

def stars(p):
    if np.isnan(p): return ''
    return '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''

print('Helper defined. Ready for analysis.')